# Aegis restoration: prepare → sanity → train → evaluate → export

This notebook is the control panel for the repository scripts, on Lightning AI
or Colab. The model, data transforms, losses, resume logic, and evaluation live
in `src/` and `scripts/`; editing them changes both notebook and terminal runs.

**Start with `STAGE = "prepare"`.** Running all cells in that stage prepares data
and weights without training. Later choose one stage and run from Settings onward.
The sanity stage is a short GPU check, excluded from research results.

First experiment: official pretrained **Restormer**, fine-tuned on RealRain-1k-H.
This is a reproducible baseline, not a proposed novel method. The 20-epoch recipe
is an initial experiment, not a claim to reproduce the original 300k-step training.

Prepare downloads on CPU; enable the GPU for sanity/training/evaluation.
Colab files are temporary: retain the run directory in persistent storage before
ending the runtime. For long runs, the terminal command works independently of
this notebook. Checkpoints resume from the last completed epoch.


## 1. Locate the updated repository
On Lightning, open this notebook inside the repository you cloned. On Colab,
clone the **updated** repository first or extract the provided code bundle under
`/content/BTP_Aegis-Perception`. A standalone copy of this notebook is insufficient.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/BTP_Aegis-Perception")]
candidates += [Path.cwd() / "BTP_Aegis-Perception"]
REPO = next(
    (p for p in candidates if (p / "configs/restoration/rain_restormer.yaml").is_file()), None
)
if REPO is None:
    raise FileNotFoundError(
        "Open the notebook in the updated repository or extract the code bundle first."
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
print("Repository:", REPO)
print("Python:", sys.executable)

In [ ]:
# Run once for this environment; this uses the notebook kernel's Python.
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[notebook,metrics,dev,export]"], check=True
)

## 2. Settings
The YAML contains architecture, data transforms, optimizer, precision, seed,
and output directory. Change experimental settings there, not in copied cells.
A100 supports the default bf16 recipe; use fp16 or fp32 on other supported GPUs.
Use a new output directory for every changed experiment. Resume requires the same
model, data, optimizer, schedule, and seed.


In [ ]:
import torch
import yaml

from aegis_perception.config import load_config, validate_restoration_config
from aegis_perception.data.catalog import sha256
from aegis_perception.data.manifest import load_manifest

STAGE = "prepare"  # prepare | sanity | train | evaluate | export
CONFIG = Path("configs/restoration/rain_restormer.yaml")
DATASET_ID = "realrain1k"
RESUME = False
EVALUATE_TEST = False  # Enable only after model/config selection is frozen.

assert STAGE in {"prepare", "sanity", "train", "evaluate", "export"}
config = load_config(CONFIG)
validate_restoration_config(config)
RUN_DIR = Path(config["run"]["output_dir"])
RUN_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST = Path(config["data"]["manifest"])
WEIGHTS = Path(config["model"]["pretrained"])


def run(script, *arguments):
    subprocess.run([sys.executable, str(script), *map(str, arguments)], check=True)


def require_gpu():
    if not torch.cuda.is_available():
        raise RuntimeError("Enable the GPU and verify that PyTorch has CUDA before this stage.")
    print("GPU:", torch.cuda.get_device_name(0))


print("Stage:", STAGE, "| PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("Config:", CONFIG, "| Main run:", RUN_DIR)

## 3. Obtain and audit the data
The reviewed archive is verified by size and SHA-256 before extraction.
Existing image pairs skip downloading; existing archives are verified and reused. This downloads only the selected dataset.
Preserve the supplied manifest: it retains the release's train/validation/test
split. Do not rebuild it to obtain a better score. The audit checks files,
geometry, IDs, and exact target duplicates; sequence/near-duplicate leakage still
requires scene review. RealRain-H and RealRain-L contain related scenes and should
not be treated as independent train/test domains.


In [ ]:
pairs_present = all(
    r.input_path.is_file() and r.target_path.is_file() for r in load_manifest(MANIFEST)
)
if STAGE == "prepare" and not pairs_present:
    run("scripts/data/download_datasets.py", "--dataset", DATASET_ID)
    run("scripts/data/extract_datasets.py", "--extract", "--dataset", DATASET_ID)
run("scripts/data/audit_manifest.py", MANIFEST)
print("Manifest SHA-256:", sha256(MANIFEST))
for split in ("train", "val", "test"):
    print(split, len(load_manifest(MANIFEST, split)))

## 4. Inspect actual training pairs
Check alignment and background sharpness. These figures preview training pairs,
not restoration results. Blurred/non-driving scenes limit what we can claim about
road perception, even if restoration metrics improve.


In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

records = random.Random(42).sample(load_manifest(MANIFEST, "train"), 4)
fig, axes = plt.subplots(4, 3, figsize=(12, 10), squeeze=False)
for axes_row, record in zip(axes, records, strict=True):
    with Image.open(record.input_path) as a, Image.open(record.target_path) as b:
        a, b = np.asarray(a.convert("RGB")), np.asarray(b.convert("RGB"))
    diff = np.clip(np.abs(a.astype(float) - b.astype(float)) * 4, 0, 255).astype("uint8")
    for axis, image, label in zip(
        axes_row, (a, b, diff), ("Rainy", "Target", "Difference x4"), strict=True
    ):
        axis.imshow(image)
        axis.set_title(f"{record.sample_id}: {label}")
        axis.axis("off")
plt.tight_layout()
fig.savefig(RUN_DIR / "data_preview.png", dpi=160)
plt.show()

## 5. Load the official pretrained model
The [official Restormer deraining release](https://github.com/swz30/Restormer/tree/main/Deraining)
links to the weights below. Loading is strict: incompatible/missing parameters
stop the run. The original network consumes RGB `[0,1]`; our wrapper converts
from/to the repository's `[-1,1]` interface. There is no random-weight fallback.
We record the downloaded weights' checksum. Architecture source revision:
`68dc6ac472db26f16361150cb7a96a1bc87da93f` (MIT license retained).


In [ ]:
import gdown

from aegis_perception.checkpoints import load_pretrained, save_checkpoint
from aegis_perception.models import build_generator

# File ID resolved from the folder linked by the official deraining README.
OFFICIAL_WEIGHTS_URL = "https://drive.google.com/uc?id=1uuejKpyo0G_5M4DAO2J9_Dijy550tjc5"
if not WEIGHTS.is_file():
    if STAGE != "prepare":
        raise FileNotFoundError("Run the prepare stage to obtain the official weights first.")
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(OFFICIAL_WEIGHTS_URL, str(WEIGHTS), quiet=False, use_cookies=False)

weights_hash = sha256(WEIGHTS)
EXPECTED_WEIGHTS_SHA256 = "225ed9acf80b3527d7080e5182918f642a23a78ef8593ad700b7e924bd660681"
if weights_hash != EXPECTED_WEIGHTS_SHA256:
    raise ValueError(
        "Weights differ from the verified official release; download deraining.pth again."
    )
model = build_generator(config["model"])
load_pretrained(model, WEIGHTS)
print("Official weights SHA-256:", weights_hash)
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
BASELINE = RUN_DIR / "pretrained.pt"
save_checkpoint(
    BASELINE,
    {
        "generator": model.state_dict(),
        "config": config,
        "epoch": 0,
        "manifest_sha256": sha256(MANIFEST),
        "pretrained_sha256": weights_hash,
        "pretrained_source": OFFICIAL_WEIGHTS_URL,
    },
)
del model  # The training subprocess loads its own model.

## 6. Optional short training check
Set `STAGE = "sanity"` for two epochs on eight fixed training patches and two
validation patches. This checks the training path at low cost. Inspect its loss,
outputs, speed, and peak memory before the full run. These subset/crop metrics
are debugging outputs and must not enter the paper's tables.


In [ ]:
import copy

SANITY_DIR = Path(str(RUN_DIR) + "_sanity")
if STAGE == "sanity":
    require_gpu()
    sanity = copy.deepcopy(config)
    sanity["run"]["name"] += "_sanity"
    sanity["run"]["output_dir"] = str(SANITY_DIR)
    sanity["data"].update(
        max_train_samples=8,
        max_val_samples=2,
        random_crop=False,
        random_flip=False,
        eval_mode="crop",
        num_workers=0,
    )
    sanity["train"].update(epochs=2, checkpoint_every=1)
    SANITY_DIR.mkdir(parents=True, exist_ok=True)
    sanity_config = SANITY_DIR / "config.yaml"
    sanity_config.write_text(yaml.safe_dump(sanity, sort_keys=False))
    resume_args = ["--resume", SANITY_DIR / "last.pt"] if RESUME else []
    run("scripts/restoration/train.py", "--config", sanity_config, *resume_args)
    print(json.loads((SANITY_DIR / "history.json").read_text()))

## 7. Full fine-tuning / resume
Set `STAGE = "train"`. Use `RESUME = True` after an interrupted run. Fresh training
refuses an existing run directory, preventing accidental overwrites.
`last.pt` contains model/optimizer/scheduler/scaler/RNG states and history;
`best.pt` is selected by mean validation RGB PSNR. Validation uses native
resolution, batch 1, with padding removed before metrics. Test is not used here.

Equivalent terminal command:
```bash
python scripts/restoration/train.py --config configs/restoration/rain_restormer.yaml
# To resume the same experiment:
python scripts/restoration/train.py --config configs/restoration/rain_restormer.yaml \
  --resume artifacts/restoration/rain_restormer_seed42/last.pt
```


In [ ]:
if STAGE == "train":
    require_gpu()
    resume_args = ["--resume", RUN_DIR / "last.pt"] if RESUME else []
    run("scripts/restoration/train.py", "--config", CONFIG, *resume_args)

## 8. Inspect the learning curve
A lower training loss is insufficient: watch validation gain over raw images,
review failures, and measure run time. The pilot's measured epoch time—not a
promised GPU-hour estimate—determines the compute budget.


In [ ]:
if (RUN_DIR / "history.json").is_file():
    history = json.loads((RUN_DIR / "history.json").read_text())
    epochs = [r["epoch"] for r in history]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(epochs, [r["train_g"] for r in history], label="Training loss")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[1].plot(epochs, [r["val_psnr_db"] for r in history], label="Restored")
    axes[1].plot(epochs, [r["val_raw_psnr_db"] for r in history], label="Raw")
    axes[1].set_ylabel("Validation RGB PSNR (dB)")
    axes[1].legend()
    for axis in axes:
        axis.set_xlabel("Epoch")
    plt.tight_layout()
    fig.savefig(RUN_DIR / "learning_curves.png", dpi=200)
    plt.show()
    print("Latest epoch:", history[-1])

## 9. Evaluate the pretrained baseline and selected model
Set `STAGE = "evaluate"`. Default: validation only. Once choices are frozen,
set `EVALUATE_TEST = True` for the held-out test comparison. Both checkpoints use
identical preprocessing and scenes. Save per-image metrics, raw PSNR/SSIM, and gains.
RGB PSNR is not directly comparable to a paper reporting Y-channel PSNR.
Native full-frame metrics are not comparable to resized/crop sanity metrics.


In [ ]:
BEST = RUN_DIR / "best.pt"
if STAGE == "evaluate":
    require_gpu()
    if not BEST.is_file():
        raise FileNotFoundError("Train the main experiment before evaluating best.pt.")
    split = "test" if EVALUATE_TEST else "val"
    for label, checkpoint in [("pretrained", BASELINE), ("finetuned", BEST)]:
        output = RUN_DIR / f"metrics_{split}_{label}.json"
        run(
            "scripts/restoration/evaluate.py",
            "--config",
            CONFIG,
            "--checkpoint",
            checkpoint,
            "--split",
            split,
            "--output",
            output,
        )
        print(label, json.loads(output.read_text())["summary"]["all"])

## 10. View fixed validation examples
This shows raw, pretrained, fine-tuned, and target views for the same predetermined
validation IDs. Keep both successes and failures. PSNR improvement alone does not
prove improved object detection; that requires labelled driving-scene evaluation.


In [ ]:
if STAGE == "evaluate":
    from aegis_perception.checkpoints import load_generator
    from aegis_perception.data.paired import normalized_tensor_to_pil, pil_to_normalized_tensor
    from aegis_perception.inference import predict

    samples = load_manifest(MANIFEST, "val")[:3]
    fig, axes = plt.subplots(len(samples), 4, figsize=(14, 8), squeeze=False)
    for i, sample in enumerate(samples):
        with Image.open(sample.input_path) as image:
            raw = image.convert("RGB")
        with Image.open(sample.target_path) as image:
            target = image.convert("RGB")
        axes[i, 0].imshow(raw)
        axes[i, 3].imshow(target)
    for column, checkpoint in ((1, BASELINE), (2, BEST)):
        generator, _ = load_generator(checkpoint, device="cuda")
        with torch.inference_mode():
            for i, sample in enumerate(samples):
                with Image.open(sample.input_path) as image:
                    inputs = pil_to_normalized_tensor(image.convert("RGB")).unsqueeze(0).cuda()
                axes[i, column].imshow(normalized_tensor_to_pil(predict(generator, inputs)[0]))
        del generator, inputs
        torch.cuda.empty_cache()
    for i, row in enumerate(axes):
        for axis, label in zip(row, ("Raw", "Pretrained", "Fine-tuned", "Target"), strict=True):
            axis.set_title(f"{samples[i].sample_id}: {label}")
            axis.axis("off")
    plt.tight_layout()
    fig.savefig(RUN_DIR / "validation_examples.png", dpi=200)
    plt.show()

## 11. Export and validate ONNX
Set `STAGE = "export"`. Export currently fixes the spatial size to the configured
patch size; numerical validation is a CPU smoke check on that shape. This is
not a full-frame TensorRT engine, INT8 model, or hardware latency claim.
Those are later measured deployment experiments. The shared image-restoration
script already supports native-resolution PyTorch inference.


In [ ]:
if STAGE == "export":
    if not BEST.is_file():
        raise FileNotFoundError("Train the main experiment before exporting best.pt.")
    onnx_path = RUN_DIR / "restormer.onnx"
    run("scripts/deployment/export_onnx.py", "--checkpoint", BEST, "--output", onnx_path)
    run("scripts/deployment/validate_onnx.py", "--checkpoint", BEST, "--onnx", onnx_path)

## Retain the experiment evidence
Keep `run.json`, `history.json`, `best.pt`, `last.pt`, metrics JSON, and fixed
figures from the run directory. Record the cloud machine's displayed billing
rate separately. Run metadata contains the resolved config, source/environment,
manifest checksum, pretrained checksum, parameter count, precision, and device.
`resume.json` records the latest resume environment. The code stores weights,
metrics, and generated figures in ignored `artifacts/`, not in Git.

Before a submission: audit scene/sequence leakage, freeze the protocol, compare
against strong baselines, repeat the final controlled comparisons with multiple
seeds, evaluate independent real driving scenes, and measure the deployed
accuracy/latency trade-off. This notebook establishes the restoration baseline;
it does not establish novelty or guarantee publication.
